# AWS for AI/ML

## Overview
Amazon Web Services (AWS) is the leading cloud platform for AI/ML workloads. It provides managed infrastructure, specialized hardware (GPUs, TPUs, Inferentia), and fully managed ML services that eliminate the need to manage infrastructure yourself.

---

## 1. Core AWS Services for AI/ML

### S3 Object Storage
Amazon S3 (Simple Storage Service) is the backbone of ML data pipelines:
- Store training datasets, model artifacts, predictions
- Versioning for data reproducibility
- Lifecycle policies for cost management
- S3 Select: query data in-place with SQL

### EC2 Compute
Key instance families for ML:

| Family | Hardware | Use Case |
|--------|----------|----------|
| p3 | NVIDIA V100 | Deep learning training |
| p4d | NVIDIA A100 | Large model training |
| g4dn | NVIDIA T4 | Inference, smaller training |
| g5 | NVIDIA A10G | Inference + training |
| inf2 | AWS Inferentia2 | Cost-optimized inference |
| trn1 | AWS Trainium | Training (up to 50% cheaper) |

### Lambda Serverless Inference
- Event-driven, pay-per-invocation
- 10GB memory, 15-min timeout
- Container image support (up to 10GB) for ML models
- Cold start challenge: use Provisioned Concurrency

### ECS/EKS Containers
- **ECS**: Managed Docker container service
- **EKS**: Managed Kubernetes use with Kubeflow for ML pipelines

---

## 2. Amazon SageMaker

SageMaker is AWS's fully managed ML platform covering the entire ML lifecycle:

```
Data Prep → Training → Tuning → Deployment → Monitoring
```

### Key Components

**SageMaker Studio** Web-based IDE for ML
- Jupyter notebooks with managed kernels
- Experiment tracking
- Model registry

**Training Jobs**
- Managed training on any instance type
- Built-in algorithms (XGBoost, Linear Learner, etc.)
- Custom containers
- Distributed training: SageMaker Data Parallel, Model Parallel

**Hyperparameter Tuning (HPO)**
- Bayesian optimization, Random search, Hyperband
- Objective: $$\theta^* = \arg\min_{\theta} \mathcal{L}_{val}(f_\theta)$$

**Endpoints (Real-time Inference)**
- Deploy models behind a REST API
- Auto-scaling based on traffic
- Multi-model endpoints (cost sharing)
- Serverless inference

**Batch Transform** Offline batch inference on S3 data

**SageMaker Pipelines** MLOps workflow automation

**Feature Store** Online (low latency) + Offline (S3) feature storage

**Model Monitor** Detect data/model drift in production

**Clarify** Bias detection and explainability (SHAP values)

---

## 3. Amazon Bedrock

Fully managed service for foundation models:
- **Anthropic Claude** (Haiku, Sonnet, Opus)
- **Meta Llama** (3, 3.1, 3.2)
- **Amazon Titan** (text, embeddings, image)
- **Mistral** (7B, Mixtral)
- **Stability AI** (image generation)

Features: Knowledge bases (RAG), Agents, Guardrails, Model evaluation

---

## 4. IAM for ML
- Create least-privilege roles for SageMaker, Lambda
- S3 bucket policies for data access
- Resource-based policies vs identity-based policies
- Use instance profiles (no hardcoded credentials)

---

## 5. Cost Optimization
- **Spot Instances**: up to 90% cheaper, use for fault-tolerant training
- **Savings Plans**: commit to usage for 1-3 years
- **Right-sizing**: use Compute Optimizer recommendations
- **S3 Intelligent-Tiering**: auto-move data to cheaper storage

---

## 6. Data Lake Architecture
```
S3 (raw) → AWS Glue (ETL) → S3 (processed) → Athena (SQL query)
                                             → SageMaker (training)
```

- **AWS Glue**: serverless ETL with PySpark
- **Athena**: query S3 data with SQL (pay-per-query)
- **Lake Formation**: governance and access control

In [1]:
# Install: pip install boto3 sagemaker
import boto3
import sagemaker
try:
    from sagemaker import get_execution_role  # SageMaker SDK v2
except ImportError:
    get_execution_role = None  # requires sagemaker<3 + AWS execution context

# ── S3 Operations ──────────────────────────────────────────────
s3 = boto3.client('s3', region_name='us-east-1')

# Upload a file
# s3.upload_file('local_file.csv', 'my-bucket', 'data/train.csv')

# Download a file
# s3.download_file('my-bucket', 'data/train.csv', 'local_train.csv')

# List objects
# response = s3.list_objects_v2(Bucket='my-bucket', Prefix='data/')
# for obj in response['Contents']:
#     print(obj['Key'], obj['Size'])

print("boto3 S3 client created (configure AWS credentials first)")

boto3 S3 client created (configure AWS credentials first)


In [2]:
# ── SageMaker Training Job ──────────────────────────────────────
import sagemaker
try:
    from sagemaker.sklearn.estimator import SKLearn  # SageMaker SDK v2
except ImportError:
    SKLearn = None  # requires sagemaker<3

# Example: train a scikit-learn model on SageMaker
# role = get_execution_role()  # IAM role with SageMaker permissions

# sklearn_estimator = SKLearn(
#     entry_point='train.py',       # Your training script
#     role=role,
#     instance_type='ml.m5.xlarge',
#     framework_version='1.2-1',
#     hyperparameters={'n_estimators': 100, 'max_depth': 5}
# )

# # Start training
# sklearn_estimator.fit({'train': 's3://my-bucket/data/train.csv'})

# # Deploy endpoint
# predictor = sklearn_estimator.deploy(
#     initial_instance_count=1,
#     instance_type='ml.t2.medium'
# )

# # Predict
# result = predictor.predict([[1.0, 2.0, 3.0]])
print("SageMaker training job template ready")

SageMaker training job template ready


In [3]:
# ── Amazon Bedrock ──────────────────────────────────────────────
import boto3
import json

# bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

# # Call Claude via Bedrock
# body = json.dumps({
#     "anthropic_version": "bedrock-2023-05-31",
#     "max_tokens": 1024,
#     "messages": [{"role": "user", "content": "Explain SageMaker in 3 sentences"}]
# })

# response = bedrock.invoke_model(
#     body=body,
#     modelId='anthropic.claude-3-sonnet-20240229-v1:0',
#     contentType='application/json',
#     accept='application/json'
# )

# result = json.loads(response['body'].read())
# print(result['content'][0]['text'])
print("Bedrock client template ready")

Bedrock client template ready


In [4]:
# ── SageMaker Pipelines ─────────────────────────────────────────
# from sagemaker.workflow.pipeline import Pipeline
# from sagemaker.workflow.steps import TrainingStep, ProcessingStep
# from sagemaker.workflow.parameters import ParameterString, ParameterFloat

# # Define pipeline parameters
# input_data = ParameterString(name='InputData', default_value='s3://bucket/data')
# learning_rate = ParameterFloat(name='LearningRate', default_value=0.01)

# # Define steps
# step_train = TrainingStep(name='TrainModel', estimator=sklearn_estimator, inputs={...})

# # Create pipeline
# pipeline = Pipeline(name='MyMLPipeline', parameters=[input_data, learning_rate], steps=[step_train])
# pipeline.upsert(role_arn=role)
# pipeline.start()
print("SageMaker Pipelines template ready")

SageMaker Pipelines template ready


## Additional Learning Resources

### Official Docs
- [AWS Machine Learning](https://aws.amazon.com/machine-learning/)
- [Amazon SageMaker Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html)
- [Amazon Bedrock Docs](https://docs.aws.amazon.com/bedrock/)
- [boto3 Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)

### Courses
- [AWS ML Specialty Certification](https://aws.amazon.com/certification/certified-machine-learning-specialty/)
- [Practical Data Science on AWS (Coursera)](https://www.coursera.org/specializations/practical-data-science)
- [AWS SageMaker Studio Lab (free)](https://studiolab.sagemaker.aws/)

### GitHub
- [Amazon SageMaker Examples](https://github.com/aws/amazon-sagemaker-examples)
- [AWS Samples ML](https://github.com/aws-samples)

### Books
- *Data Science on AWS* Chris Fregly & Antje Barth (O'Reilly)